In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.flow_matching.helpers import choose_device
from src.flow_matching.loader import (
    load_model_info,
    load_trained_model,
    prob_path_from_config,
)
from src.flow_matching.neural_posterior import NeuralPosterior
from src.flow_matching.simulator import Model as BatchedModel

device = choose_device()

# Loading the trained FM model

In [ ]:
# fill in desired job_id or directory name 
job_id = "15055686"

# loading the model (found via job_id, or path)
vector_field, classifier, config = load_trained_model(job_id, device, ema=False)

# load probability path used in training
path = prob_path_from_config(config)

# load useful training parameters
model_info = load_model_info(config, path=path, device=device)

In [ ]:
inf_params  = model_info['inf_params']
N           = model_info['N']
vector_dim  = model_info['vector_dim']
burstparams = model_info['burstparams']
mean        = model_info['mean']
std         = model_info['std']
transdimensional = model_info['transdimensional']
YBKG        = config['path']['p_data']['init_params']['model_params']['ybkg']

prior = path.p_data.prior

# Simulating a burst to do inference on
We simulate a burst that we would like to infer the defining parameter values from. 

In [ ]:
# sample lightcurve to condition on
params, simulated_counts, components = path.p_data.sample(1)
burstparams = prior.samples_as_dict(params)
N_true = components[0][0].item()
print("This example has N = ", components[0][0].item(), " components")
print(
    f"""
    The burst parameters are:
    t0 =\t {np.array_str(burstparams['t0'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    log(amp) =\t {np.array_str(burstparams['amp'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    log(rise) =\t {np.array_str(burstparams['rise'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    skew =\t {np.array_str(burstparams['skew'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    """
)
simulated_counts = simulated_counts[0]
plt.plot(simulated_counts.cpu().numpy())
plt.show()

# Using the model to generate a FM posterior 

## Sampling FM posterior $p^\phi(\theta \mid y, N$)

In [ ]:
learned_posterior = NeuralPosterior(
    vector_field,
    classifier,
    model_info,
    path,
    device=device
)

In [ ]:
num_samples = 1000
samples_per_batch = 1000

Ns, theta_samples = learned_posterior.sample(num_samples, simulated_counts, samples_per_batch, time_steps=200)
p_N = learned_posterior.p_N
Ns = Ns.to('cpu')
theta_samples

## classifier results

In [ ]:
plt.bar(range(1, len(p_N.cpu().detach().numpy())+1), height=p_N.cpu().detach().numpy())
plt.xlabel("$N_{pred}$")
plt.title('p(N|y)')

print("True number of components is: ", components[0][0].item())
print("Most likely estimate from classifier: ", range(1, len(p_N.cpu().detach().numpy())+1)[torch.where(p_N == torch.max(p_N))[0]])

# plotting the samples

In [ ]:
# convert sampled parameter vectors to dictionary
posterior_param_samples = path.p_data.prior.samples_as_dict(theta_samples)

# simulate burst profiles from sampled parameters and components
posterior_curve_samples = BatchedModel(
    device=device, 
    time=torch.linspace(0, 1, simulated_counts.shape[0]), burstparams=posterior_param_samples, ybkg=YBKG, ncomp=Ns
        ).get_flux()

import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(11, 4))
gs = gridspec.GridSpec(1, 2, width_ratios=[1.5, 1])  # first subplot wider

ax1 = plt.subplot(gs[0])
ax2 = plt.subplot(gs[1])

ax1.plot(np.linspace(0, 1, simulated_counts.shape[0]), simulated_counts.flatten().cpu(),'k-', alpha=0.6,  label='observed flux')
sample_colors=['pink', 'yellow', 'orange', 'purple', 'green', 'cyan', 'red', 'grey', 'magenta', 'blue']

# plot mean curve
mean_profile = torch.mean(posterior_curve_samples[0:num_samples], dim=0).cpu()
time_ = np.linspace(0, 1, simulated_counts.shape[0])
ax1.plot(time_, mean_profile, color='white', label='mean profile', linewidth=1)

# plot 95th percentile
q1 = torch.quantile(posterior_curve_samples, 0.025, dim=0).cpu()
q3 = torch.quantile(posterior_curve_samples, 0.975, dim=0).cpu()
iqr = q3 - q1
ax1.fill_between(time_, q1, q3, color='red', alpha=1, label='std profile')

ax1.grid(linestyle='dotted')
ax1.set_ylabel('counts', fontsize=14)
ax1.set_xlabel('t', fontsize=14)
ax1.set_xlim(0,1)
ax1.tick_params(axis='both', labelsize=14)

ax2.bar(
    range(1, len(p_N.cpu().detach().numpy())+1), 
    height=p_N.cpu().detach().numpy(), 
    color='lightsteelblue', 
    edgecolor='black',    
    linewidth=1,
    label=range(1, N+1)
    )
ax2.set_xlabel("$N$", fontsize=14)
ax2.set_ylabel('$p(N\mid y)$', fontsize=14, labelpad=15)
ax2.tick_params(axis='both', labelsize=14)

# integer tick labels for histogram
import matplotlib.ticker as mticker
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()